In [2]:
import cv2
import mediapipe as mp
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

# For webcam input:
cap = cv2.VideoCapture(0)
with mp_hands.Hands(
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # To improve performance, optionally mark the image as not writeable to
    # pass by reference.
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(image)

    # Draw the hand annotations on the image.
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    if results.multi_hand_landmarks:
      for hand_landmarks in results.multi_hand_landmarks:
        mp_drawing.draw_landmarks(
            image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())
    # Flip the image horizontally for a selfie-view display.
    cv2.imshow('MediaPipe Hands', cv2.flip(image, 1))
    if (cv2.waitKey(5) & 0xFF) == ord('q'):
      break
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)  #

-1

In [9]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

def get_landmark_list(frame, hand_landmarks):
    h, w, _ = frame.shape
    lm_list = []
    for idx, lm in enumerate(hand_landmarks.landmark):
        cx, cy = int(lm.x * w), int(lm.y * h)   # normalized → pixels
        lm_list.append([idx, cx, cy])
    return lm_list


def fingers_up(lm_list):
    """Returns a list of 5 ints [thumb, index, middle, ring, pinky], 1 = up."""
    tips = [4, 8, 12, 16, 20]
    fingers = []

    # Thumb: compare x (sideways). Assumes a right hand on a mirrored feed.
    fingers.append(1 if lm_list[4][1] > lm_list[3][1] else 0)

    # Other four fingers: compare y (tip above PIP joint = up)
    for tip in tips[1:]:
        fingers.append(1 if lm_list[tip][2] < lm_list[tip - 2][2] else 0)

    return fingers


def classify_gesture(fingers):
    if fingers == [0, 0, 0, 0, 0]:
        return "FIST"
    if fingers == [1, 1, 1, 1, 1]:
        return "OPEN PALM"
    if fingers == [0, 1, 0, 0, 0]:
        return "POINTING"
    if fingers == [0, 1, 1, 0, 0]:
        return "PEACE"
    if fingers == [1, 0, 0, 0, 0]:
        return "THUMBS UP"
    return "UNKNOWN"


hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,             # one hand is plenty for this project
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)                       # mirror, feels natural
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)     # BGR → RGB for MediaPipe
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            lm_list = get_landmark_list(frame, hand_landmarks)
            if lm_list:
                fingers = fingers_up(lm_list)
                gesture = classify_gesture(fingers)
                print(fingers, gesture)
                cv2.putText(frame, gesture, (20, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

    cv2.imshow("Hand Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

[0, 0, 0, 0, 0] FIST
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 0, 0, 0, 0] FIST
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0, 0, 0] POINTING
[0, 1, 0

In [10]:
import cv2
import mediapipe as mp
import math

def _distance(lm_list, p1, p2):
    """Euclidean pixel distance between two landmark indices."""
    x1, y1 = lm_list[p1][1], lm_list[p1][2]
    x2, y2 = lm_list[p2][1], lm_list[p2][2]
    return math.hypot(x2 - x1, y2 - y1)

def normalized_distance(lm_list, p1, p2):
    """Distance between p1 and p2 as a ratio of hand size (camera-distance invariant)."""
    ref = distance(lm_list, 0, 9)   # wrist to middle knuckle = hand scale
    if ref == 0:
        return 0
    return distance(lm_list, p1, p2) / ref

def _hand_scale(lm_list):
    ref = _distance(lm_list, 0, 9)
    return ref if ref != 0 else 1

def thumb_up(lm_list, hand_label):
    if hand_label == "Right":
        return lm_list[4][1] > lm_list[3][1]   # tip x to the right of joint
    else:
        return lm_list[4][1] < lm_list[3][1]
    
def fingers_up(lm_list, hand_label="Left"):
    """Returns [thumb, index, middle, ring, pinky], 1 = up."""
    fingers = []

    # Thumb compares x; direction depends on handedness
    if hand_label == "Left":
        fingers.append(1 if lm_list[4][1] > lm_list[3][1] else 0)
    else:
        fingers.append(1 if lm_list[4][1] < lm_list[3][1] else 0)

    # Other four compare y (tip above PIP joint = up)
    for tip in [8, 12, 16, 20]:
        fingers.append(1 if lm_list[tip][2] < lm_list[tip - 2][2] else 0)

    return fingers

def classify(lm_list, hand_label="Right"):
    """Main entry point. Returns a gesture name string."""
    if not lm_list or len(lm_list) < 21:
        return "NONE"

    fingers = fingers_up(lm_list, hand_label)
    scale = _hand_scale(lm_list)

    # Distance-based gestures take priority (more specific)
    pinch = _distance(lm_list, 4, 8) / scale       # thumb tip to index tip
    if pinch < 0.3 and fingers[2] and fingers[3] and fingers[4]:
        return "OK"                                 # thumb+index circle, others up

    # Finger-count gestures
    if fingers == [0, 0, 0, 0, 0]:
        return "FIST"
    if fingers == [1, 1, 1, 1, 1]:
        return "OPEN_PALM"
    if fingers == [0, 1, 0, 0, 0]:
        return "POINT"
    if fingers == [0, 1, 1, 0, 0]:
        return "PEACE"
    if fingers == [1, 0, 0, 0, 0]:
        return "THUMB"

    return "UNKNOWN"

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)

cap = cv2.VideoCapture(0)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    
    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    

    gesture = "NONE"
    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        label = "Right"
        if results.multi_handedness:
            label = results.multi_handedness[0].classification[0].label

        h, w, _ = frame.shape
        lm_list = [[i, int(lm.x * w), int(lm.y * h)]
                   for i, lm in enumerate(hand.landmark)]

        gesture =  classify(lm_list, label)
        mp_draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)

    cv2.putText(frame, gesture, (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 2)
    cv2.imshow("Gesture Test", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()